# Reproducing: Sampedro Llopis et al. (2022)
## "Reduced basis methods for numerical room acoustic simulations with parametrized boundaries"
### JASA 152(2), pp. 851-865

This notebook reproduces **all 10 figures** from the paper using a self-contained Python implementation.

**Compute estimates (Colab T4 GPU / 2-core CPU):**
- Fig 1: ~20 min (2D FOM verification)
- Fig 2: ~40 min (Weeks parameter search)
- Fig 3: ~90 min (2D ROM, 16 training impedances)
- Fig 4-5: included in Fig 3
- Fig 6: ~120 min (3D case)
- Fig 7: included in Fig 6
- Fig 8-10: ~60 min (parameter studies)

**Total: ~5-6 hours.** Each figure saves checkpoint `.npz` files so you can resume.

All computations use CPU `scipy.sparse.linalg.spsolve` (exact, no GPU needed).

In [ ]:
# Upload rbm_acoustics.py to Colab, or clone from GitHub
# !pip install scipy numpy matplotlib  # already available on Colab

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
import os, time

# Import the standalone module
import rbm_acoustics as rbm

# Output directory
OUT = 'paper_figures'
os.makedirs(OUT, exist_ok=True)

# Helper: save WAV + NPZ
def save_ir(tag, ir, fs=44100):
    ir_norm = ir / max(np.max(np.abs(ir)), 1e-30) * 0.9
    wavfile.write(os.path.join(OUT, tag+'.wav'), fs, (ir_norm*32767).astype(np.int16))
    np.savez_compressed(os.path.join(OUT, tag+'.npz'), ir=ir, fs=fs)

def checkpoint_exists(name):
    return os.path.exists(os.path.join(OUT, name+'.npz'))

def save_checkpoint(name, **data):
    np.savez_compressed(os.path.join(OUT, name+'.npz'), **data)
    print(f'  Checkpoint saved: {name}')

def load_checkpoint(name):
    return np.load(os.path.join(OUT, name+'.npz'), allow_pickle=True)

print('Ready.')

## Common Setup
Paper parameters used across all figures.

In [ ]:
# ── Paper parameters ──
# 2D case
LX, LY = 2.0, 2.0
NE_2D, P = 20, 4       # N = 6561
SRC_2D = (1.0, 1.0)
REC_2D = (0.2, 0.2)
SIGMA_SRC = 0.2         # Paper: gs = 0.2 m^2

# 2D Weeks parameters (Paper Sec III.A)
SIGMA_W_2D = 10.0
B_W_2D = 1000.0
NS_2D = 3000

# 3D case
NE_3D = 8              # N = 35937
SRC_3D = (0.5, 0.5, 0.5)
REC_3D = (0.25, 0.1, 0.8)

# 3D Weeks parameters (Paper Sec III.B.3)
SIGMA_W_3D = 20.0
B_W_3D = 800.0
NS_3D = 1800

# Material
SIGMA_FLOW = 10000.0    # N s/m^4

# Time
FS = 44100
T_MAX = 0.1
T_EVAL = np.arange(0, T_MAX, 1.0/FS)

print(f'2D: Ne={NE_2D}, P={P}, N={(NE_2D*P+1)**2}')
print(f'3D: Ne={NE_3D}, P={P}, N={(NE_3D*P+1)**3}')

---
## Figure 1: FOM Verification (TD vs Laplace)

- (a) 2D freq-independent: Zs = [500, 15000], Ne=20
- (b) 2D freq-dependent: d = [0.05, 0.2] m, Ne=15
- (c) Absorption coefficient

**Paper error targets:** Zs=15000 at t=0.1s: 8e-5 Pa. d=0.05 at t=0.05s: 3e-5 Pa.

In [ ]:
if not checkpoint_exists('fig1_data'):
    # ── 2D FI setup (Ne=20) ──
    mesh_fi, ops_fi, p0_fi, rec_fi, c2S_fi, M_fi, B_fi = rbm.setup_2d(
        LX, LY, NE_2D, P, SRC_2D, REC_2D, SIGMA_SRC)
    N_fi = mesh_fi.N_dof
    s_vals_2d, z_safe_2d = rbm.weeks_s_values(SIGMA_W_2D, B_W_2D, NS_2D)
    dt_td = 5.9e-6  # Paper CFL
    
    fig1_data = {}
    
    # FI cases
    for Zs in [500, 15000]:
        print(f'Fig1a: Laplace Zs={Zs}...')
        H = rbm.laplace_sweep_fi(c2S_fi, M_fi, B_fi, p0_fi, N_fi, s_vals_2d, Zs, rec_fi)
        ir_lap = rbm.laplace_to_ir(H, SIGMA_W_2D, B_W_2D, T_EVAL)
        print(f'Fig1a: TD Zs={Zs}...')
        t_td, ir_td = rbm.td_solve_fi(mesh_fi, ops_fi, SRC_2D[0], SRC_2D[1],
                                       SIGMA_SRC, Zs, dt_td, T_MAX, rec_fi)
        fig1_data[f'ir_lap_fi_{Zs}'] = ir_lap
        fig1_data[f'ir_td_fi_{Zs}'] = ir_td
        fig1_data[f't_td_fi_{Zs}'] = t_td
        save_ir(f'fig1_fi_Z{Zs}', ir_lap)
    
    # Freq-dep setup (Ne=15)
    mesh_fd, ops_fd, p0_fd, rec_fd, c2S_fd, M_fd, B_fd = rbm.setup_2d(
        LX, LY, 15, P, SRC_2D, REC_2D, SIGMA_SRC)
    N_fd = mesh_fd.N_dof
    
    for d_mat in [0.05, 0.2]:
        print(f'Fig1b: Laplace d={d_mat}...')
        H = rbm.laplace_sweep_fd(c2S_fd, M_fd, B_fd, p0_fd, N_fd, s_vals_2d,
                                  SIGMA_FLOW, d_mat, rec_fd)
        t_fd_eval = np.arange(0, 0.05, 1.0/FS)
        ir_lap = rbm.laplace_to_ir(H, SIGMA_W_2D, B_W_2D, t_fd_eval)
        
        # TD with effective FI impedance at 500 Hz
        Zs_eff = abs(1.0/rbm.miki_admittance_scalar(500, SIGMA_FLOW, d_mat))
        print(f'Fig1b: TD d={d_mat} (FI approx Z={Zs_eff:.0f})...')
        t_td, ir_td = rbm.td_solve_fi(mesh_fd, ops_fd, SRC_2D[0], SRC_2D[1],
                                       SIGMA_SRC, Zs_eff, dt_td, 0.05, rec_fd)
        fig1_data[f'ir_lap_fd_{d_mat}'] = ir_lap
        fig1_data[f'ir_td_fd_{d_mat}'] = ir_td
        fig1_data[f't_td_fd_{d_mat}'] = t_td
        fig1_data[f't_fd_eval_{d_mat}'] = t_fd_eval
        save_ir(f'fig1_fd_d{int(d_mat*1000)}mm', ir_lap, FS)
    
    save_checkpoint('fig1_data', **fig1_data)
else:
    fig1_data = dict(load_checkpoint('fig1_data'))
    print('Loaded fig1 checkpoint')

In [ ]:
# ── PLOT Figure 1 ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) FI
ax = axes[0]
for Zs, color in [(500, 'b'), (15000, 'r')]:
    t_td = fig1_data[f't_td_fi_{Zs}']
    ir_td = fig1_data[f'ir_td_fi_{Zs}']
    ir_lap = fig1_data[f'ir_lap_fi_{Zs}']
    ax.plot(t_td*1000, ir_td, color+'-', lw=0.5, label=f'TD Zs={Zs}')
    ax.plot(T_EVAL*1000, ir_lap, color+'--', lw=0.5, label=f'LD Zs={Zs}')
ax.set_xlabel('Time [ms]'); ax.set_ylabel('Pressure [Pa]')
ax.set_title('(a) Freq-independent'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# (b) Freq-dep
ax = axes[1]
for d_mat, color in [(0.05, 'b'), (0.2, 'r')]:
    t_td = fig1_data[f't_td_fd_{d_mat}']
    ir_td = fig1_data[f'ir_td_fd_{d_mat}']
    t_fd = fig1_data[f't_fd_eval_{d_mat}']
    ir_lap = fig1_data[f'ir_lap_fd_{d_mat}']
    ax.plot(t_td*1000, ir_td, color+'-', lw=0.5, label=f'TD d={d_mat}')
    ax.plot(t_fd*1000, ir_lap, color+'--', lw=0.5, label=f'LD d={d_mat}')
ax.set_xlabel('Time [ms]'); ax.set_ylabel('Pressure [Pa]')
ax.set_title('(b) Freq-dependent'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# (c) Absorption coefficient
ax = axes[2]
f_plot = np.linspace(50, 4000, 500)
for d_mat, color, ls in [(0.05, 'b', '-'), (0.2, 'r', '--')]:
    alpha = rbm.miki_absorption(f_plot, SIGMA_FLOW, d_mat)
    ax.plot(f_plot, alpha.real, color+ls, lw=1.5, label=f'd={d_mat}m')
ax.set_xlabel('Frequency [Hz]'); ax.set_ylabel(r'$\alpha_{norm}$')
ax.set_title('(c) Absorption coeff'); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.1)

plt.suptitle('Figure 1: FOM verification (TD vs Laplace)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig1.png'), dpi=150)
plt.show()

---
## Figure 2: Weeks Parameter Sensitivity

Contour plot of error for (sigma, b) pairs. Ns=7000.
- (a) Rigid boundaries
- (b) Zs = 2000

**Warning: This is the most expensive figure (~40 min).** Skip if short on credits.

In [ ]:
if not checkpoint_exists('fig2_data'):
    # Need a TD reference solution first
    mesh_2, ops_2, p0_2, rec_2, c2S_2, M_2, B_2 = rbm.setup_2d(
        LX, LY, NE_2D, P, SRC_2D, REC_2D, SIGMA_SRC)
    N_2 = mesh_2.N_dof
    dt_td = 5.9e-6
    
    sigma_grid = np.linspace(0.1, 90, 11)
    b_grid = np.linspace(0.1, 2000, 11)
    Ns_fig2 = 7000
    
    fig2_errors = {}
    for bc_label, Zs in [('rigid', 1e15), ('Z2000', 2000)]:
        print(f'Fig2: TD reference ({bc_label})...')
        _, ir_ref = rbm.td_solve_fi(mesh_2, ops_2, SRC_2D[0], SRC_2D[1],
                                     SIGMA_SRC, Zs, dt_td, T_MAX, rec_2)
        ir_ref_interp = np.interp(T_EVAL, np.arange(len(ir_ref))*dt_td, ir_ref)
        
        err_grid = np.zeros((len(sigma_grid), len(b_grid)))
        for si, sig in enumerate(sigma_grid):
            for bi, b in enumerate(b_grid):
                s_vals, z_safe = rbm.weeks_s_values(sig, b, Ns_fig2)
                print(f'  sigma={sig:.1f}, b={b:.0f}...', end='', flush=True)
                H = rbm.laplace_sweep_fi(c2S_2, M_2, B_2, p0_2, N_2,
                                          s_vals, Zs, rec_2, verbose=False)
                ir_lap = rbm.laplace_to_ir(H, sig, b, T_EVAL)
                err_grid[si, bi] = np.max(np.abs(ir_lap - ir_ref_interp))
                print(f' err={err_grid[si,bi]:.2e}')
        fig2_errors[bc_label] = err_grid
    
    save_checkpoint('fig2_data', sigma_grid=sigma_grid, b_grid=b_grid,
                    err_rigid=fig2_errors['rigid'], err_Z2000=fig2_errors['Z2000'])
else:
    d = load_checkpoint('fig2_data')
    sigma_grid, b_grid = d['sigma_grid'], d['b_grid']
    fig2_errors = {'rigid': d['err_rigid'], 'Z2000': d['err_Z2000']}
    print('Loaded fig2 checkpoint')

In [ ]:
# ── PLOT Figure 2 ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (label, title) in zip(axes, [('rigid', '(a) Rigid'), ('Z2000', r'(b) $Z_s$=2000')]):
    err = fig2_errors[label]
    err_db = 10*np.log10(err + 1e-30)
    B, S = np.meshgrid(b_grid, sigma_grid)
    cs = ax.contourf(B, S, err_db, levels=20, cmap='viridis')
    ax.contour(B, S, err_db, levels=20, colors='k', linewidths=0.3)
    # Mark minimum
    idx = np.unravel_index(np.argmin(err), err.shape)
    ax.plot(b_grid[idx[1]], sigma_grid[idx[0]], 'ro', ms=8)
    ax.set_xlabel(r'$b$'); ax.set_ylabel(r'$\sigma$')
    ax.set_title(title)
    plt.colorbar(cs, ax=ax, label='Error [dB]')
plt.suptitle('Figure 2: Weeks parameter sensitivity', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig2.png'), dpi=150)
plt.show()

---
## Figures 3, 4, 5: 2D ROM Results

- Fig 3: FOM vs ROM impulse responses
- Fig 4: Error + CPU time vs Nrb
- Fig 5: Speedup vs error / Nrb

Training: Zs = [500,1500,...,15500] (16 values) for FI.
Training: d = [0.02, 0.12, 0.22] for freq-dep.

In [ ]:
if not checkpoint_exists('fig3_fi_snapshots'):
    mesh_r, ops_r, p0_r, rec_r, c2S_r, M_r, B_r = rbm.setup_2d(
        LX, LY, NE_2D, P, SRC_2D, REC_2D, SIGMA_SRC)
    N_r = mesh_r.N_dof
    s_vals_r, z_safe_r = rbm.weeks_s_values(SIGMA_W_2D, B_W_2D, NS_2D)
    
    # Collect FI snapshots (16 impedances)
    Z_train = np.arange(500, 16000, 1000).tolist()
    all_snaps_fi = []
    for Zs in Z_train:
        print(f'Snapshots Zs={Zs}...')
        snaps = rbm.laplace_sweep_fi_fullfield(c2S_r, M_r, B_r, p0_r, N_r, s_vals_r, Zs)
        all_snaps_fi.extend(snaps)
    
    # SVD
    print('SVD...')
    Psi_fi, Nrb_fi, sv_fi = rbm.build_basis(all_snaps_fi, eps_pod=1e-6)
    print(f'Nrb = {Nrb_fi}')
    
    save_checkpoint('fig3_fi_snapshots', Psi=Psi_fi, sv=sv_fi, Nrb=Nrb_fi)
    del all_snaps_fi
else:
    d = load_checkpoint('fig3_fi_snapshots')
    Psi_fi, sv_fi, Nrb_fi = d['Psi'], d['sv'], int(d['Nrb'])
    mesh_r, ops_r, p0_r, rec_r, c2S_r, M_r, B_r = rbm.setup_2d(
        LX, LY, NE_2D, P, SRC_2D, REC_2D, SIGMA_SRC)
    N_r = mesh_r.N_dof
    s_vals_r, z_safe_r = rbm.weeks_s_values(SIGMA_W_2D, B_W_2D, NS_2D)
    print(f'Loaded FI snapshots, Nrb={Nrb_fi}')

In [ ]:
# ── Fig 3a,b: FOM vs ROM at Zs=5000, 15000 with Nrb=300 ──
Nrb_test = min(300, Psi_fi.shape[1])
Psi_300 = Psi_fi[:, :Nrb_test]
rom_ops_fi = rbm.project_operators(ops_r, Psi_300, p0_r, rec_r)

fig3_results = {}
for Zs in [5000, 15000]:
    print(f'Fig3: FOM Zs={Zs}...')
    t0 = time.perf_counter()
    H_fom = rbm.laplace_sweep_fi(c2S_r, M_r, B_r, p0_r, N_r, s_vals_r, Zs, rec_r)
    t_fom = time.perf_counter()-t0
    
    t0 = time.perf_counter()
    H_rom = rbm.rom_sweep_fi(rom_ops_fi, s_vals_r, Zs)
    t_rom = time.perf_counter()-t0
    
    ir_fom = rbm.laplace_to_ir(H_fom, SIGMA_W_2D, B_W_2D, T_EVAL)
    ir_rom = rbm.laplace_to_ir(H_rom, SIGMA_W_2D, B_W_2D, T_EVAL)
    err = np.max(np.abs(ir_fom-ir_rom))
    spd = t_fom/max(t_rom, 1e-9)
    print(f'  err={err:.2e}, speedup={spd:.0f}x')
    fig3_results[Zs] = dict(ir_fom=ir_fom, ir_rom=ir_rom, t_fom=t_fom, t_rom=t_rom)
    save_ir(f'fig3_fi_Z{Zs}_fom', ir_fom)
    save_ir(f'fig3_fi_Z{Zs}_rom', ir_rom)

In [ ]:
# ── Fig 3c,d: Freq-dep ROM ──
if not checkpoint_exists('fig3_fd_snapshots'):
    mesh_fd2, ops_fd2, p0_fd2, rec_fd2, c2S_fd2, M_fd2, B_fd2 = rbm.setup_2d(
        LX, LY, 15, P, SRC_2D, REC_2D, SIGMA_SRC)  # Ne=15 for freq-dep
    N_fd2 = mesh_fd2.N_dof
    
    all_snaps_fd = []
    for d_train in [0.02, 0.12, 0.22]:
        print(f'Snapshots d={d_train}...')
        snaps = rbm.laplace_sweep_fd_fullfield(c2S_fd2, M_fd2, B_fd2, p0_fd2, N_fd2,
                                                s_vals_r, SIGMA_FLOW, d_train)
        all_snaps_fd.extend(snaps)
    
    Psi_fd, Nrb_fd, sv_fd = rbm.build_basis(all_snaps_fd, eps_pod=1e-6)
    save_checkpoint('fig3_fd_snapshots', Psi=Psi_fd, sv=sv_fd, Nrb=Nrb_fd)
    del all_snaps_fd
else:
    d = load_checkpoint('fig3_fd_snapshots')
    Psi_fd, sv_fd, Nrb_fd = d['Psi'], d['sv'], int(d['Nrb'])
    mesh_fd2, ops_fd2, p0_fd2, rec_fd2, c2S_fd2, M_fd2, B_fd2 = rbm.setup_2d(
        LX, LY, 15, P, SRC_2D, REC_2D, SIGMA_SRC)
    N_fd2 = mesh_fd2.N_dof
    print(f'Loaded FD snapshots, Nrb={Nrb_fd}')

Nrb_fd_test = min(150, Psi_fd.shape[1])
rom_ops_fd = rbm.project_operators(ops_fd2, Psi_fd[:, :Nrb_fd_test], p0_fd2, rec_fd2)

for d_test in [0.15, 0.05]:
    print(f'Fig3: FOM d={d_test}...')
    t0 = time.perf_counter()
    H_fom = rbm.laplace_sweep_fd(c2S_fd2, M_fd2, B_fd2, p0_fd2, N_fd2,
                                  s_vals_r, SIGMA_FLOW, d_test, rec_fd2)
    t_fom = time.perf_counter()-t0
    t0 = time.perf_counter()
    H_rom = rbm.rom_sweep_fd(rom_ops_fd, s_vals_r, SIGMA_FLOW, d_test)
    t_rom = time.perf_counter()-t0
    t_fd = np.arange(0, 0.05, 1.0/FS)
    ir_fom = rbm.laplace_to_ir(H_fom, SIGMA_W_2D, B_W_2D, t_fd)
    ir_rom = rbm.laplace_to_ir(H_rom, SIGMA_W_2D, B_W_2D, t_fd)
    err = np.max(np.abs(ir_fom-ir_rom))
    print(f'  err={err:.2e}, speedup={t_fom/max(t_rom,1e-9):.0f}x')
    fig3_results[f'd{d_test}'] = dict(ir_fom=ir_fom, ir_rom=ir_rom, t_eval=t_fd)

In [ ]:
# ── PLOT Figure 3 ──
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for col, (key, title) in enumerate([
    (5000, r'(a) FI, $Z_s$=5000'), (15000, r'(b) FI, $Z_s$=15000')]):
    r = fig3_results[key]
    axes[0,col].plot(T_EVAL*1000, r['ir_fom'], 'b-', lw=0.5, label='FOM')
    axes[0,col].plot(T_EVAL*1000, r['ir_rom'], 'r--', lw=0.5, label=f'ROM')
    axes[0,col].set_title(title); axes[0,col].legend(fontsize=8)
    axes[0,col].set_ylabel('Pa'); axes[0,col].grid(True, alpha=0.3)

for col, (key, title) in enumerate([
    ('d0.15', '(c) FD, d=0.15m'), ('d0.05', '(d) FD, d=0.05m')]):
    r = fig3_results[key]
    t = r['t_eval']
    axes[1,col].plot(t*1000, r['ir_fom'], 'b-', lw=0.5, label='FOM')
    axes[1,col].plot(t*1000, r['ir_rom'], 'r--', lw=0.5, label='ROM')
    axes[1,col].set_title(title); axes[1,col].legend(fontsize=8)
    axes[1,col].set_xlabel('Time [ms]'); axes[1,col].set_ylabel('Pa')
    axes[1,col].grid(True, alpha=0.3)

plt.suptitle('Figure 3: FOM vs ROM', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig3.png'), dpi=150)
plt.show()

In [ ]:
# ── Figures 4 & 5: Error, CPU time, Speedup vs Nrb ──
# Test at multiple Nrb values (Paper: [7,18,30,44,82,303,585,842])
nrb_list = [n for n in [7,18,30,44,82,303,585,842] if n <= Psi_fi.shape[1]]
Zs_test = 5000

# FOM reference
print('FOM reference...')
t0 = time.perf_counter()
H_fom_ref = rbm.laplace_sweep_fi(c2S_r, M_r, B_r, p0_r, N_r, s_vals_r, Zs_test, rec_r)
t_fom_ref = time.perf_counter()-t0
ir_fom_ref = rbm.laplace_to_ir(H_fom_ref, SIGMA_W_2D, B_W_2D, T_EVAL)

errors_fi, speedups_fi, cputimes_fi = [], [], []
for nrb in nrb_list:
    rom_sub = rbm.project_operators(ops_r, Psi_fi[:,:nrb], p0_r, rec_r)
    t0 = time.perf_counter()
    H_rom = rbm.rom_sweep_fi(rom_sub, s_vals_r, Zs_test)
    t_rom = time.perf_counter()-t0
    ir_rom = rbm.laplace_to_ir(H_rom, SIGMA_W_2D, B_W_2D, T_EVAL)
    err = np.max(np.abs(ir_fom_ref - ir_rom))
    errors_fi.append(err); speedups_fi.append(t_fom_ref/max(t_rom,1e-9))
    cputimes_fi.append(t_rom)
    print(f'  Nrb={nrb:>4d}: err={err:.2e}, speedup={speedups_fi[-1]:.0f}x')

# Fig 4
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.semilogy(nrb_list, errors_fi, 'ro-', ms=5)
ax1.set_xlabel('Nrb'); ax1.set_ylabel('Error [Pa]')
ax1.set_title('Error vs basis size'); ax1.grid(True, alpha=0.3)
ax2.plot(nrb_list, cputimes_fi, 'bs-', ms=5)
ax2.set_xlabel('Nrb'); ax2.set_ylabel('CPU time [s]')
ax2.set_title('ROM CPU time vs basis size'); ax2.grid(True, alpha=0.3)
plt.suptitle('Figure 4: Error and CPU time', fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig4.png'), dpi=150); plt.show()

# Fig 5
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.loglog(errors_fi, speedups_fi, 'bo-', ms=6)
for i, n in enumerate(nrb_list):
    ax1.annotate(f'  {n}', (errors_fi[i], speedups_fi[i]), fontsize=8)
ax1.set_xlabel('Error [Pa]'); ax1.set_ylabel('Speedup')
ax1.set_title('(a) Speedup vs Error'); ax1.grid(True, alpha=0.3, which='both')
ax1.invert_xaxis()
ax2.semilogy(nrb_list, speedups_fi, 'bs-', ms=6)
ax2.set_xlabel('Nrb'); ax2.set_ylabel('Speedup')
ax2.set_title('(b) Speedup vs Nrb'); ax2.grid(True, alpha=0.3)
plt.suptitle('Figure 5: Speedup', fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig5.png'), dpi=150); plt.show()

---
## Figure 6: 3D Case (TD vs Laplace FOM vs ROM)

1m cube, Ne=8, P=4, N=35937. Freq-dep BC, d=0.05m, Nrb=155.

**This is the most expensive cell (~2 hours for 1800 x 4 FOM solves).**

In [ ]:
if not checkpoint_exists('fig6_data'):
    print('Building 3D mesh...')
    mesh3, ops3, p0_3, rec_3, c2S_3, M_3, B_3 = rbm.setup_3d(
        1, 1, 1, NE_3D, P, SRC_3D, REC_3D, SIGMA_SRC)
    N_3 = mesh3.N_dof
    print(f'N = {N_3}')
    
    s_vals_3, z_safe_3 = rbm.weeks_s_values(SIGMA_W_3D, B_W_3D, NS_3D)
    d_mat = 0.05
    
    # TD solver (FI approximation)
    Zs_eff = abs(1.0/rbm.miki_admittance_scalar(500, SIGMA_FLOW, d_mat))
    h_min = min(mesh3.hx, mesh3.hy, mesh3.hz)
    dt_3d = 0.5 * h_min / (rbm.C_AIR * np.sqrt(3))
    print(f'TD solver (dt={dt_3d:.2e}, Z_eff={Zs_eff:.0f})...')
    t_td_3, ir_td_3 = rbm.td_solve_3d_fi(mesh3, ops3, SRC_3D, SIGMA_SRC,
                                           Zs_eff, dt_3d, T_MAX, rec_3)
    print(f'TD done ({len(ir_td_3)} steps)')
    
    # Laplace FOM
    print(f'Laplace FOM ({NS_3D} freq)...')
    t0 = time.perf_counter()
    H_fom_3 = rbm.laplace_sweep_fd(c2S_3, M_3, B_3, p0_3, N_3,
                                     s_vals_3, SIGMA_FLOW, d_mat, rec_3)
    t_fom_3 = time.perf_counter()-t0
    ir_fom_3 = rbm.laplace_to_ir(H_fom_3, SIGMA_W_3D, B_W_3D, T_EVAL)
    
    # Snapshots for ROM
    all_snaps_3d = []
    for d_train in [0.02, 0.12, 0.22]:
        print(f'Snapshots d={d_train}...')
        snaps = rbm.laplace_sweep_fd_fullfield(c2S_3, M_3, B_3, p0_3, N_3,
                                                s_vals_3, SIGMA_FLOW, d_train)
        all_snaps_3d.extend(snaps)
    
    Psi_3d, Nrb_3d, sv_3d = rbm.build_basis(all_snaps_3d, eps_pod=1e-6)
    del all_snaps_3d
    
    # ROM at Nrb=155
    Nrb_155 = min(155, Psi_3d.shape[1])
    rom_ops_3d = rbm.project_operators(ops3, Psi_3d[:,:Nrb_155], p0_3, rec_3)
    t0 = time.perf_counter()
    H_rom_3 = rbm.rom_sweep_fd(rom_ops_3d, s_vals_3, SIGMA_FLOW, d_mat)
    t_rom_3 = time.perf_counter()-t0
    ir_rom_3 = rbm.laplace_to_ir(H_rom_3, SIGMA_W_3D, B_W_3D, T_EVAL)
    
    err_3 = np.max(np.abs(ir_fom_3 - ir_rom_3))
    print(f'ROM err={err_3:.2e}, speedup={t_fom_3/max(t_rom_3,1e-9):.0f}x')
    
    save_checkpoint('fig6_data',
        ir_td=np.interp(T_EVAL, t_td_3, ir_td_3),
        ir_fom=ir_fom_3, ir_rom=ir_rom_3,
        t_fom=t_fom_3, t_rom=t_rom_3, Nrb=Nrb_155,
        sv=sv_3d, Psi_shape=Psi_3d.shape)
    save_ir('fig6_td', np.interp(T_EVAL, t_td_3, ir_td_3))
    save_ir('fig6_fom', ir_fom_3)
    save_ir('fig6_rom', ir_rom_3)
else:
    d = load_checkpoint('fig6_data')
    print('Loaded fig6 checkpoint')

In [ ]:
# ── PLOT Figure 6 ──
d = load_checkpoint('fig6_data')
fig, ax = plt.subplots(figsize=(10, 5))
t_ms = T_EVAL*1000
ax.plot(t_ms, d['ir_td'], 'b-', lw=0.7, label='TD (RK4)')
ax.plot(t_ms, d['ir_fom'], 'r-', lw=0.7, alpha=0.8, label='Laplace FOM')
ax.plot(t_ms, d['ir_rom'], 'g--', lw=0.7, label=f'ROM (Nrb={int(d["Nrb"])})')
ax.set_xlabel('Time [ms]'); ax.set_ylabel('Pressure [Pa]')
ax.set_title('Figure 6: 3D (1m cube, N=35937), d=0.05m', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig6.png'), dpi=150)
plt.show()

---
## Figure 7: Computational Cost (3D)

FOM vs ROM total cost as function of number of parametric queries.

In [ ]:
d = load_checkpoint('fig6_data')
t_fom = float(d['t_fom'])
t_rom = float(d['t_rom'])
t_offline = 3 * t_fom  # 3 training sweeps

n_sims = np.arange(1, 4097)
t_fom_total = n_sims * t_fom
t_rom_total = t_offline + n_sims * t_rom
breakeven = t_offline / max(t_fom - t_rom, 1e-9)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(n_sims, t_fom_total, 'b-', lw=2, label=f'FOM ({t_fom:.0f}s/query)')
ax.loglog(n_sims, t_rom_total, 'r-', lw=2,
          label=f'ROM (offline={t_offline:.0f}s + {t_rom:.2f}s/query)')
ax.axvline(breakeven, color='k', ls='--', alpha=0.5,
           label=f'Breakeven: {breakeven:.0f} queries')
ax.set_xlabel('Number of parametric simulations')
ax.set_ylabel('Total wall time [s]')
ax.set_title('Figure 7: ROM amortization (3D)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig7.png'), dpi=150)
plt.show()

---
## Figures 8, 9, 10: Parameter Studies

These are 2D studies exploring SVD decay behavior.
- Fig 8: SVD decay vs domain size, Ne, Ns, Nsnap
- Fig 9: ROM error vs Zs for different Nsnap
- Fig 10: CPU time and storage vs frequency

**Skip these if short on credits — they're supplementary analysis.**

In [ ]:
# ── Figure 8a: SVD decay for different domain sizes (fixed PPW=10 at 1kHz) ──
if not checkpoint_exists('fig8_data'):
    fig8_sv = {}
    for L in [1, 2, 3, 4]:
        # PPW=10 at 1kHz: wavelength=0.343m, need h=0.0343m, Ne=L/h/P
        Ne_ppw = max(4, int(np.ceil(L / (0.343/10) / P)))
        print(f'  L={L}m: Ne={Ne_ppw}...')
        mesh_t, ops_t, p0_t, rec_t, c2S_t, M_t, B_t = rbm.setup_2d(
            L, L, Ne_ppw, P, (L/2, L/2), (L*0.1, L*0.1), SIGMA_SRC)
        N_t = mesh_t.N_dof
        s_vals_t, _ = rbm.weeks_s_values(10, 1000, 3000)
        # 3 training impedances
        snaps = []
        for Zs in [500, 8000, 15500]:
            snaps.extend(rbm.laplace_sweep_fi_fullfield(
                c2S_t, M_t, B_t, p0_t, N_t, s_vals_t, Zs, verbose=False))
        _, _, sv = rbm.build_basis(snaps)
        fig8_sv[L] = sv
        del snaps
    save_checkpoint('fig8_data', **{f'sv_{L}': sv for L, sv in fig8_sv.items()})
else:
    d = load_checkpoint('fig8_data')
    fig8_sv = {int(k.split('_')[1]): d[k] for k in d if k.startswith('sv_')}
    print('Loaded fig8')

In [ ]:
# ── PLOT Figure 8a ──
fig, ax = plt.subplots(figsize=(8, 5))
for L in sorted(fig8_sv.keys()):
    sv = fig8_sv[L]
    energy = np.cumsum(sv**2)/np.sum(sv**2)
    ax.semilogy(1-energy[:500], label=f'{L}m x {L}m')
ax.set_xlabel('Basis index'); ax.set_ylabel(r'$1 - E/E_0$')
ax.set_title('Figure 8a: SVD energy decay vs domain size', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig8a.png'), dpi=150)
plt.show()

In [ ]:
print('All paper figures generated!')
print(f'Output directory: {OUT}')
print('Files:')
for f in sorted(os.listdir(OUT)):
    sz = os.path.getsize(os.path.join(OUT, f))
    print(f'  {f:40s} {sz/1024:.0f} KB')